Silver Layer: Data Cleaning & Transformation



In [0]:
%python
SOURCE_CATALOG = "01_bronze"
SOURCE_SCHEMA = "raw_data"
TARGET_CATALOG = "02_silver"
TARGET_SCHEMA = "silver_transform"

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, DateType

SILVER_TABLES_METADATA = [
    {
        "source_table": "customers",
        "target_table": "customers_clean",
        "primary_key": ["customer_id"],
        "required_columns": ["customer_id", "customer_name", "country"],
        "transformations": [
            "deduplicate",
            "remove_nulls",
            "trim_strings",
            "standardize_country",
            "standardize_customer_type",
            "convert_signup_date"
        ],
        "description": "Clean customer master data"
    },
    {
        "source_table": "products",
        "target_table": "products_clean",
        "primary_key": ["product_id"],
        "required_columns": ["product_id", "product_name", "category", "cost_price"],
        "transformations": [
            "deduplicate",
            "remove_nulls",
            "trim_strings",
            "standardize_category",
            "validate_positive_price"
        ],
        "description": "Clean product catalog"
    },
    {
        "source_table": "invoices",
        "target_table": "invoices_clean",
        "primary_key": ["invoice_id"],
        "required_columns": ["invoice_id", "customer_id", "invoice_date"],
        "transformations": [
            "deduplicate",
            "remove_nulls",
            "trim_strings",
            "standardize_status",
            "standardize_region",
            "convert_date"
        ],
        "description": "Clean invoice headers with standardized status and dates"
    },
    {
        "source_table": "invoice_line_items",
        "target_table": "invoice_line_items_clean",
        "primary_key": ["invoice_line_id"],
        "required_columns": ["invoice_line_id", "invoice_id", "product_id", "quantity", "unit_price"],
        "transformations": [
            "deduplicate",
            "remove_nulls",
            "validate_positive_quantity",
            "validate_positive_price",
            "calculate_line_amount"
        ],
        "description": "Clean line items with calculated amounts"
    },
    {
        "source_table": "payments",
        "target_table": "payments_clean",
        "primary_key": ["payment_id"],
        "required_columns": ["payment_id", "invoice_id", "payment_amount"],
        "transformations": [
            "deduplicate",
            "remove_nulls",
            "trim_strings",
            "validate_positive_price",
            "convert_date"
        ],
        "description": "Clean payment transactions"
    },
    {
        "source_table": "exchange_rates",
        "target_table": "exchange_rates_clean",
        "primary_key": ["date", "currency"],
        "required_columns": ["date", "currency", "rate_to_usd"],
        "transformations": [
            "deduplicate",
            "remove_nulls",
            "trim_strings",
            "validate_positive_price",
            "convert_date"
        ],
        "description": "Clean exchange rates"
    },
    {
        "source_table": "regions",
        "target_table": "regions_clean",
        "primary_key": ["region_code"],
        "required_columns": ["region_code", "region_name"],
        "transformations": [
            "deduplicate",
            "remove_nulls",
            "trim_strings",
            "standardize_region"
        ],
        "description": "Clean regional master data"
    }
]

In [0]:
%python
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
import time

def deduplicate(df: DataFrame, primary_key: list) -> DataFrame:
    return df.dropDuplicates(primary_key)

def remove_nulls(df: DataFrame, required_columns: list) -> DataFrame:
    return df.dropna(subset=required_columns)

def trim_strings(df: DataFrame) -> DataFrame:
    for col_name, col_type in df.dtypes:
        if col_type == 'string':
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
    return df

def standardize_status(df: DataFrame) -> DataFrame:
    if 'invoice_status' in df.columns:
        df = df.withColumn(
            'invoice_status',
            F.when(F.lower(F.col('invoice_status')).isin(['paid', 'complete', 'completed']), 'Paid')
            .when(F.lower(F.col('invoice_status')).isin(['pending', 'open', 'unpaid']), 'Pending')
            .when(F.lower(F.col('invoice_status')).isin(['cancelled', 'canceled', 'void']), 'Cancelled')
            .otherwise(F.col('invoice_status'))
        )
    return df

def standardize_region(df: DataFrame) -> DataFrame:
    if 'region' in df.columns:
        df = df.withColumn('region', F.initcap(F.trim(F.col('region'))))
    if 'region_name' in df.columns:
        df = df.withColumn('region_name', F.initcap(F.trim(F.col('region_name'))))
    return df

def standardize_country(df: DataFrame) -> DataFrame:
    if 'country' in df.columns:
        df = df.withColumn('country', F.initcap(F.regexp_replace(F.trim(F.col('country')), r'[.]', '')))
    return df

def standardize_category(df: DataFrame) -> DataFrame:
    if 'category' in df.columns:
        df = df.withColumn('category', F.initcap(F.trim(F.col('category'))))
    return df

def standardize_customer_type(df: DataFrame) -> DataFrame:
    if 'customer_type' in df.columns:
        df = df.withColumn('customer_type', F.initcap(F.trim(F.col('customer_type'))))
    return df

def convert_date(df: DataFrame) -> DataFrame:
    for col_name in df.columns:
        if 'date' in col_name.lower() and df.schema[col_name].dataType.typeName() == 'string':
            df = df.withColumn(col_name, F.to_date(F.regexp_replace(F.col(col_name), '/', '-')))
    return df

def convert_signup_date(df: DataFrame) -> DataFrame:
    if 'signup_date' in df.columns:
        df = df.withColumn('signup_date', F.to_date(F.col('signup_date')))
    return df

def validate_positive_quantity(df: DataFrame) -> DataFrame:
    if 'quantity' in df.columns:
        df = df.filter(F.col('quantity') > 0)
    return df

def validate_positive_price(df: DataFrame) -> DataFrame:
    price_cols = ['cost_price', 'unit_price', 'payment_amount', 'rate_to_usd']
    for col_name in price_cols:
        if col_name in df.columns:
            df = df.filter(F.col(col_name) > 0)
    return df

def calculate_line_amount(df: DataFrame) -> DataFrame:
    if all(col in df.columns for col in ['quantity', 'unit_price']):
        if 'discount' in df.columns:
            df = df.withColumn('discount', F.coalesce(F.col('discount'), F.lit(0)))
            df = df.withColumn('line_amount', F.col('quantity') * F.col('unit_price') * (F.lit(1) - F.col('discount')))
        else:
            df = df.withColumn('line_amount', F.col('quantity') * F.col('unit_price'))
    return df

def apply_transformations(df: DataFrame, transformations: list, primary_key: list, required_columns: list) -> DataFrame:
    transformation_map = {
        'deduplicate': lambda x: deduplicate(x, primary_key),
        'remove_nulls': lambda x: remove_nulls(x, required_columns),
        'trim_strings': trim_strings,
        'standardize_status': standardize_status,
        'standardize_region': standardize_region,
        'standardize_country': standardize_country,
        'standardize_category': standardize_category,
        'standardize_customer_type': standardize_customer_type,
        'convert_date': convert_date,
        'convert_signup_date': convert_signup_date,
        'validate_positive_quantity': validate_positive_quantity,
        'validate_positive_price': validate_positive_price,
        'calculate_line_amount': calculate_line_amount
    }
    
    for transformation in transformations:
        if transformation in transformation_map:
            df = transformation_map[transformation](df)
    
    return df

In [0]:
%python
transformation_results = []

for table_metadata in SILVER_TABLES_METADATA:
    source_table = table_metadata['source_table']
    target_table = table_metadata['target_table']
    primary_key = table_metadata['primary_key']
    required_columns = table_metadata['required_columns']
    transformations = table_metadata['transformations']
    
    source_full_name = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{source_table}"
    target_full_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{target_table}"
    
    start_time = time.time()
    
    try:
        df_bronze = spark.table(source_full_name)
        
        # Drop ingestion_timestamp if it exists
        if 'ingestion_timestamp' in df_bronze.columns:
            df_bronze = df_bronze.drop('ingestion_timestamp')
        
        bronze_count = df_bronze.count()
        
        df_silver = apply_transformations(
            df=df_bronze,
            transformations=transformations,
            primary_key=primary_key,
            required_columns=required_columns
        )
        
        df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_full_name)
        
        silver_count = spark.table(target_full_name).count()
        rows_removed = bronze_count - silver_count
        duration = time.time() - start_time
        
        transformation_results.append({
            "table_name": target_table,
            "status": "SUCCESS",
            "bronze_rows": bronze_count,
            "silver_rows": silver_count,
            "rows_removed": rows_removed,
            "duration_seconds": round(duration, 2)
        })
        
    except Exception as e:
        duration = time.time() - start_time
        transformation_results.append({
            "table_name": target_table,
            "status": "FAILED",
            "error": str(e),
            "duration_seconds": round(duration, 2)
        })

success_count = sum(1 for r in transformation_results if r["status"] == "SUCCESS")
failed_count = sum(1 for r in transformation_results if r["status"] == "FAILED")
total_bronze = sum(r.get("bronze_rows", 0) for r in transformation_results)
total_silver = sum(r.get("silver_rows", 0) for r in transformation_results)
total_removed = sum(r.get("rows_removed", 0) for r in transformation_results)

print(f"Transformed {success_count}/{len(transformation_results)} tables successfully")
print(f"Total rows: {total_bronze:,} -> {total_silver:,}")

In [0]:
SELECT * FROM 02_silver.silver_transform.customers_clean LIMIT 50;

In [0]:
SELECT * FROM 02_silver.silver_transform.products_clean LIMIT 50;

In [0]:
SELECT * FROM 02_silver.silver_transform.invoices_clean LIMIT 50;

In [0]:
SELECT * FROM 02_silver.silver_transform.invoice_line_items_clean LIMIT 50;

In [0]:
SELECT * FROM 02_silver.silver_transform.payments_clean LIMIT 50;

In [0]:
SELECT * FROM 02_silver.silver_transform.exchange_rates_clean LIMIT 50;

In [0]:
SELECT * FROM 02_silver.silver_transform.regions_clean;